In [1]:
import pandas as pd
df = pd.read_csv(r"C:\Users\Kunny\Documents\GitHub\CAGI\EvoStructCLIP\FireProtDB\fireprotdb_ddg_final_all_ver2.csv", low_memory=False)

In [2]:
df_mega = df[df["SOURCE_DATASET"]=="MegaScale"].copy()

In [3]:
print(df_mega)

       PROTEIN_ID WT  POS MUT  DDG_mean SOURCE_DATASET UNIPROTKB PDB_ID  \
0            1A0N  A   12   C -1.271679      MegaScale       NaN    NaN   
1            1A0N  A   12   D -2.513681      MegaScale       NaN    NaN   
2            1A0N  A   12   E -2.120722      MegaScale       NaN    NaN   
3            1A0N  A   12   F -2.238222      MegaScale       NaN    NaN   
4            1A0N  A   12   G -0.142810      MegaScale       NaN    NaN   
...           ... ..  ...  ..       ...            ...       ...    ...   
347020    v2_6IVS  Y    7   R  0.056541      MegaScale       NaN    NaN   
347021    v2_6IVS  Y    7   S -0.514305      MegaScale       NaN    NaN   
347022    v2_6IVS  Y    7   T -0.211927      MegaScale       NaN    NaN   
347023    v2_6IVS  Y    7   V -0.062451      MegaScale       NaN    NaN   
347024    v2_6IVS  Y    7   W  0.223641      MegaScale       NaN    NaN   

       MEGASCALE  
0           1A0N  
1           1A0N  
2           1A0N  
3           1A0N  
4   

In [4]:
import os
import pandas as pd

# 1. PDB 저장 경로 설정 (사용자 경로)
PDB_DIR = r"C:\Users\Kunny\Documents\GitHub\CAGI\EvoStructCLIP\FireProtDB\AlphaFold_model_PDBs"

# 2. 파일 경로 생성 함수
def get_corrected_pdb_path(mega_id):
    # | 기호를 _로 치환 (예: EA|run2 -> EA_run2)
    clean_name = str(mega_id).replace('|', '_')
    path = os.path.join(PDB_DIR, f"{clean_name}.pdb")
    
    if os.path.exists(path):
        return path
    return None

# 3. 유니크한 MEGASCALE ID 기준으로 파일 존재 여부 확인 (효율성)
unique_mega = pd.DataFrame(df_mega['MEGASCALE'].unique(), columns=['MEGASCALE'])
unique_mega['FILE_PATH'] = unique_mega['MEGASCALE'].apply(get_corrected_pdb_path)

# 4. 원본 df_mega에 경로 정보 병합
df_mega = pd.merge(df_mega, unique_mega, on='MEGASCALE', how='left')

# 5. 기초 통계 확인
found_count = df_mega['FILE_PATH'].notna().sum()
print(f"전체 로우: {len(df_mega)} 개")
print(f"로컬에서 찾은 PDB 파일: {found_count} 개 ({(found_count/len(df_mega))*100:.2f}%)")

전체 로우: 342675 개
로컬에서 찾은 PDB 파일: 342675 개 (100.00%)


In [5]:
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import three_to_one, is_aa

def verify_mega_mapping(row):
    if pd.isna(row['FILE_PATH']):
        return "No_File", None
    
    target_pos = int(row['POS'])
    target_wt = row['WT']
    
    try:
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure('protein', row['FILE_PATH'])
        model = structure[0]
        
        # 모든 체인을 돌며 해당 POS 확인
        for chain in model:
            if target_pos in chain:
                res = chain[target_pos]
                if is_aa(res):
                    res_name_1 = three_to_one(res.get_resname())
                    if res_name_1 == target_wt:
                        return "Match", chain.id
        
        return "Mismatch_or_Not_Found", None
    except Exception as e:
        return f"Error: {str(e)}", None

# 적용 (수량이 많으므로 진행 상황 확인을 위해 tqdm 추천)
# !pip install tqdm
from tqdm import tqdm
tqdm.pandas()

print("MegaScale WT-POS 매핑 검증 시작...")
# 유니크한 (MEGASCALE, POS, WT) 조합에 대해서만 먼저 돌리면 훨씬 빠릅니다.
unique_check = df_mega[['MEGASCALE', 'POS', 'WT', 'FILE_PATH']].drop_duplicates()
unique_check[['MAPPING_STATUS', 'MATCHED_CHAIN']] = unique_check.progress_apply(
    lambda x: pd.Series(verify_mega_mapping(x)), axis=1
)

# 결과를 다시 df_mega에 병합
df_mega = pd.merge(df_mega, unique_check, on=['MEGASCALE', 'POS', 'WT', 'FILE_PATH'], how='left')

MegaScale WT-POS 매핑 검증 시작...


100%|██████████| 19254/19254 [06:29<00:00, 49.43it/s]


In [6]:
# 1. 'Match'인 것만 추출
df_mega_cleaned = df_mega[df_mega['MAPPING_STATUS'] == 'Match'].copy()

# 2. 결과 리포트
print("=== [ MegaScale 정제 결과 ] ===")
print(f"최종 남은 로우: {len(df_mega_cleaned):,} 개")
print(f"제거된 로우: {len(df_mega) - len(df_mega_cleaned):,} 개")

# 3. 실패 원인 통계
print("\n[ 실패 원인 분포 ]")
print(df_mega['MAPPING_STATUS'].value_counts())

=== [ MegaScale 정제 결과 ] ===
최종 남은 로우: 341,443 개
제거된 로우: 1,232 개

[ 실패 원인 분포 ]
MAPPING_STATUS
Match                    341443
Mismatch_or_Not_Found      1232
Name: count, dtype: int64


In [7]:
df_mega[df_mega['MAPPING_STATUS']!='Match']["MEGASCALE"].value_counts()

MEGASCALE
2HBB    98
2B88    62
1PGA    60
1UBQ    54
2K5H    38
2KYB    38
1H8K    37
2KZI    36
5FWB    36
1MJC    36
1PGX    36
2LXK    36
1A0N    35
2LYP    35
2MKX    35
5ECA    35
2KT8    34
2ROT    34
2K52    34
1BK2    34
1CSQ    33
1S1N    33
2KRS    32
2ZW1    31
2LSS    30
1OPS    28
1EM7    26
1SIF    25
2PTL    24
1GB4    21
2M7O    19
6NS8    19
1YEZ    19
1SF0    17
5XR0    11
1UCS     9
2LX2     5
5GU9     4
2MLB     3
Name: count, dtype: int64

In [37]:
df_mega[(df_mega["MEGASCALE"]=="2HBB")]

,PROTEIN_ID,WT,POS,MUT,DDG_mean,SOURCE_DATASET,UNIPROTKB,PDB_ID,MEGASCALE,FILE_PATH,MAPPING_STATUS,MATCHED_CHAIN
71127,2HBB,A,22,C,-0.153565,MegaScale,NaN,NaN,2HBB,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,Match,A
71128,2HBB,A,22,D,0.425834,MegaScale,NaN,NaN,2HBB,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,Match,A
71129,2HBB,A,22,F,-0.546964,MegaScale,NaN,NaN,2HBB,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,Match,A
71130,2HBB,A,22,G,-0.438680,MegaScale,NaN,NaN,2HBB,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,Match,A
71131,2HBB,A,22,H,-0.095823,MegaScale,NaN,NaN,2HBB,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,Match,A
...,...,...,...,...,...,...,...,...,...,...,...,...
71977,2HBB,Y,25,R,-1.459033,MegaScale,NaN,NaN,2HBB,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,Match,A
71978,2HBB,Y,25,S,-1.754541,MegaScale,NaN,NaN,2HBB,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,Match,A
71979,2HBB,Y,25,T,-1.554375,MegaScale,NaN,NaN,2HBB,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,Match,A
71980,2HBB,Y,25,V,-1.105048,MegaScale,NaN,NaN,2HBB,C:\Users\Kunny\Documents\GitHub\CAGI\EvoStruct...,Match,A


In [29]:
# 1. 단백질별 매핑 성공률 계산
protein_stats = df_mega.groupby('MEGASCALE')['MAPPING_STATUS'].apply(
    lambda x: (x == 'Match').mean()
).reset_index(name='success_rate')

# 2. 신뢰할 수 없는 단백질 리스트 (예: 성공률 95% 미만)
THRESHOLD = 1.0
bad_proteins = protein_stats[protein_stats['success_rate'] < THRESHOLD]['MEGASCALE'].unique()

print(f"❌ 신뢰도 미달로 퇴출될 단백질 수: {len(bad_proteins)} 개")
print(f"퇴출 목록 샘플: {bad_proteins[:10]}")

# 3. 최종 필터링
# - 퇴출 단백질에 속하지 않으면서
# - 개별 Row 상태도 'Match'인 것만 유지
df_mega_ultra_clean = df_mega[
    (~df_mega['MEGASCALE'].isin(bad_proteins)) & 
    (df_mega['MAPPING_STATUS'] == 'Match')
].copy()

print(f"✅ 최종 생존 Row: {len(df_mega_ultra_clean):,} 개")

❌ 신뢰도 미달로 퇴출될 단백질 수: 39 개
퇴출 목록 샘플: ['1A0N' '1BK2' '1CSQ' '1EM7' '1GB4' '1H8K' '1MJC' '1OPS' '1PGA' '1PGX']
✅ 최종 생존 Row: 301,709 개


In [30]:
# 1. 단백질별 매핑 성공률 계산
protein_stats = df_mega.groupby('MEGASCALE')['MAPPING_STATUS'].apply(
    lambda x: (x == 'Match').mean()
).reset_index(name='success_rate')

# 2. 신뢰할 수 없는 단백질 리스트 (예: 성공률 95% 미만)
THRESHOLD = 0.95
bad_proteins = protein_stats[protein_stats['success_rate'] < THRESHOLD]['MEGASCALE'].unique()

print(f"❌ 신뢰도 미달로 퇴출될 단백질 수: {len(bad_proteins)} 개")
print(f"퇴출 목록 샘플: {bad_proteins[:10]}")

# 3. 최종 필터링
# - 퇴출 단백질에 속하지 않으면서
# - 개별 Row 상태도 'Match'인 것만 유지
df_mega_ultra_clean = df_mega[
    (~df_mega['MEGASCALE'].isin(bad_proteins)) & 
    (df_mega['MAPPING_STATUS'] == 'Match')
].copy()

print(f"✅ 최종 생존 Row: {len(df_mega_ultra_clean):,} 개")

❌ 신뢰도 미달로 퇴출될 단백질 수: 3 개
퇴출 목록 샘플: ['1PGA' '2B88' '2HBB']
✅ 최종 생존 Row: 338,995 개
